## **Predictive Analysis of Credit Card Defaults**

### Objective: 
- Dataset of customers' default payments.
- The primary goal is to predict which credit card clients are likely to default using various data mining methods.

### Background: 
Traditional risk management models classify clients as either credible or not credible based on their likelihood of default. This project aims to refine this classification by identifying specific individuals who are likely to default, enhancing the precision of credit risk assessments.

Target variable
- default.payment.next.month: Default payment (1=yes, 0=no)

The dataset contains the following features

1. ID: ID of each client
2. LIMIT_BAL: Amount of given credit in dollars (includes individual and family/supplementary credit
3. SEX: Gender (1=male, 2=female)
4. EDUCATION: (1=graduate school, 2=university, 3=high school, 4=others, 5=unknown, 6=unknown)
5. MARRIAGE: Marital status (1=married, 2=single, 3=others)
6. AGE: Age in years
7. PAY_0: Repayment status in September (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, ... 8=payment delay for eight months, 9=payment delay for nine months and above)
8. PAY_2: Repayment status in August (scale same as above)
9. PAY_3: Repayment status in July, (scale same as above)
10. PAY_4: Repayment status in June (scale same as above)
11. PAY_5: Repayment status in May (scale same as above)
12. PAY_6: Repayment status in April (scale same as above)
13. BILL_AMT1: Amount of bill statement in September (dollars)  
14. BILL_AMT2: Amount of bill statement in August (dollars)  
15. BILL_AMT3: Amount of bill statement in July (dollars)  
16. BILL_AMT4: Amount of bill statement in June (dollars)  
17. BILL_AMT5: Amount of bill statement in May (dollars)  
18. BILL_AMT6: Amount of bill statement in April (dollars)   
19. PAY_AMT1: Amount of previous payment in September (dollars)  
20. PAY_AMT2: Amount of previous payment in August (dollars)  
21. PAY_AMT3: Amount of previous payment in July (dollars)   
22. PAY_AMT4: Amount of previous payment in June (dollars)  
23. PAY_AMT5: Amount of previous payment in May (dollars)   
24. PAY_AMT6: Amount of previous payment in April (dollars)  



## 1. Reading the dataset

In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

In [2]:
# load the dataset
df = pd.read_excel('Dataset/credit_data.xlsx')

# create a pandas dataframe contining the first 10,000 rows from the credit card dataset adn drop the "ID" column
df = df.drop(columns = ["ID"]).loc[:9999]

# print the dataframe info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   LIMIT_BAL                   10000 non-null  int64  
 1   SEX                         9617 non-null   float64
 2   EDUCATION                   9617 non-null   float64
 3   MARRIAGE                    9617 non-null   float64
 4   AGE                         9617 non-null   float64
 5   PAY_0                       9617 non-null   float64
 6   PAY_2                       9617 non-null   float64
 7   PAY_3                       9617 non-null   float64
 8   PAY_4                       9617 non-null   float64
 9   PAY_5                       9617 non-null   float64
 10  PAY_6                       9642 non-null   float64
 11  BILL_AMT1                   9642 non-null   float64
 12  BILL_AMT2                   9642 non-null   float64
 13  BILL_AMT3                   9642

### Classification of features

**Table: Classification of features**

|Variable Kind|Number of Features|Feature Names
| --- | --- | --- |
| Numerical | 14 | LIMIT_BAL, AGE, BILL_AMT1, BILL_AMT2, BILL_AMT3, BILL_AMT4, BILL_AMT5, BILL_AMT6, PAY_AMT1, PAY_AMT2, PAY_AMT3, PAY_AMT4, PAY_AMT5, PAY_AMT6 |
| Ordinal  |  7| EDUCATION, PAY_0, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6 |
| Nominal  | 2 | SEX, MARRIAGE 


**Description:**

After removing the `ID` column, the dataset contains 23 features relevant to predicting credit card defaults.

- There are 14 numeric features, such as `LIMIT_BAL` (credit limit), `AGE` and many bill and payment amounts from April to September. These are continuous variables that reflect the financial behavior of clients.

- 7 features are classified as ordinal, including the six `PAY_` variables (`PAY_0`, `PAY_2` to `PAY_6`), which indicate repayment status on a monthly scale, with higher values corresponding to longer payment delays. 
    + `EDUCATION` is also treated as ordinal, as the values `1` (graduate school), `2` (university) and `3` (high school) follow a logical order. However, this feature includes categories `4` (others) and `5` – `6` (unknown) creates ambiguity, as these do not clearly fit into a ranked structure. To preserve the ordinal nature of the variable in analysis, these categories (`5` and `6`) might be grouped together as `4` (others) during preprocessing.

- The remaining 2 features are nominal: `SEX` and `MARRIAGE` . These represent categorical data with no intrinsic order and should be encoded accordingly.

This classification guides the choice of appropriate preprocessing strategies and modeling techniques, helping ensure that the data is interpreted correctly based on its underlying structure.


In [3]:
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,50000,1.0,2.0,2.0,25.0,0.0,0.0,-1.0,0.0,0.0,...,14699.0,14999.0,497.0,0.0,14805.0,294.0,300.0,498.0,1003.0,0
1,210000,1.0,2.0,2.0,30.0,-1.0,-1.0,-1.0,-1.0,0.0,...,1252.0,626.0,626.0,3980.0,7509.0,1252.0,0.0,626.0,626.0,0
2,50000,2.0,2.0,1.0,28.0,0.0,0.0,0.0,0.0,0.0,...,20304.0,20319.0,20330.0,2209.0,1726.0,710.0,724.0,733.0,712.0,0
3,20000,1.0,2.0,2.0,34.0,0.0,0.0,-2.0,-2.0,-1.0,...,-1.0,309.0,9525.0,0.0,0.0,0.0,310.0,9500.0,800.0,0
4,280000,1.0,1.0,1.0,35.0,0.0,0.0,0.0,0.0,2.0,...,177595.0,153181.0,145852.0,101585.0,90762.0,50119.0,282.0,50436.0,54184.0,0


## 2. Data Cleaning and Transformation

### 2.1 Handling missing values

In [4]:
# number of missing vaules in each feature
missing_values = df.isnull().sum()
print("Missing values: \n", missing_values)

Missing values: 
 LIMIT_BAL                       0
SEX                           383
EDUCATION                     383
MARRIAGE                      383
AGE                           383
PAY_0                         383
PAY_2                         383
PAY_3                         383
PAY_4                         383
PAY_5                         383
PAY_6                         358
BILL_AMT1                     358
BILL_AMT2                     358
BILL_AMT3                     358
BILL_AMT4                     358
BILL_AMT5                     368
BILL_AMT6                     368
PAY_AMT1                      368
PAY_AMT2                      368
PAY_AMT3                       10
PAY_AMT4                       10
PAY_AMT5                      290
PAY_AMT6                      290
default payment next month      0
dtype: int64


The output shows that several variables in the dataset contain missing values, which need to be addressed before training model:

- Repayment status features (`PAY_0`, `PAY_2`, `PAY_3`, `PAY_4`, `PAY_5`, `PAY_6`) have between 358 and 383 missing entries. Since these variables help us understand how timely clients are with their payments and are ranked in severity, therefore, they should be imputed carefully, ideally in a way that maintains their order.

- Bill statement amounts features (`BILL_AMT1` to `BILL_AMT6`) have moderate missing entries, ranging from 358 to 368 values. These are continuous numeric variables and are typically imputed using the mean.

- Payment amount features (`PAY_AMT1` to `PAY_AMT6`) show varying levels of missing entries, with some as low as 10 and others up to 368. These numeric variables should be imputed consistently to avoid having bias.

- Demographic and categorical features such as `SEX`, `AGE`, `MARRIAGE` and `EDUCATION` each have 383 missing entries. Given their nominal or ordinal nature, mode imputation would be the most appropriate strategy.

- It is also worth noting that both the target variable (`default payment next month`) and a key numeric predictor `LIMIT_BAL`, have no missing values. This ensures the target is fully usable for supervised learning and that one of the most informative features is complete.

In summary, the missing values are non-trivial and span across multiple important predictors. Addressing them properly will help ensure the model is both reliable and accurate.

In [5]:
# create a new dataframe for the cleaning process and preserve the original dataframe
df_cleaned = df.copy()

# create a subset of the dataframe containing only the numerical variables
df_numerical = df_cleaned[['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 
                   'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6','PAY_AMT1',
                   'PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']]

# get the mean values of the numerical variables and fill the missing values with the mean values
mean_values = df_numerical.mean()
df_numerical = df_numerical.fillna(mean_values)

# update the dataframe with the cleaned numerical variables
df_cleaned[df_numerical.columns] = df_numerical

In [6]:
# create a subset of the dataframe containing only the categorical variables
df_categorical = df[['SEX', 'EDUCATION', 'MARRIAGE', 
                     'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']]

# get the mode values of the categorical variables and fill the missing values with the mode values
mode_values = df_categorical.mode().iloc[0]
df_categorical = df_categorical.fillna(mode_values)

# update the dataframe with the cleaned categorical variables
df_cleaned[df_categorical.columns] = df_categorical

Data imputation is the process of replacing missing data with substituted values. This step ensures the dataset is completed and ready for model training, preventing errors and potential biases that could arise from null values.

Depending on the variable types, two methods were used to impute missing values:
- Mean value was used for all numberical variables (for example `AGE`, `BILL_AMT` and `PAY_AMT`), as these varaibles are continuous and replacing missing values with the average is statistically appropriate and maintains the overall distribution.

- Mode value was used for nominal and ordinal variables (for example `SEX`, `EDUCATION`, `MARRIAGE` and repayment status variables like `PAY_0`, `PAY_2`, etc), since these variabbles represent discrete groups. Filling in the most frequent category helps preserve the variable’s interpretability and structure.

These decisions were guided by the earlier features classification, ensuring that the imputation approach aligned with the nature of each variable. This structured strategy supports both the integrity of the dataset and the performance of the predictive models built on it.

### 2.2 Data Transpormation

In [7]:
# print value_count of SEX column
sex_freq = df_cleaned['SEX'].value_counts()
print(sex_freq)

# Add dummy variable for SEX column to df
df_cleaned = pd.get_dummies(df_cleaned, columns = ['SEX'], dtype = int, drop_first =True)

# Rename the new variable to SEX_FEMALE
df_cleaned = df_cleaned.rename(columns = {'SEX_2.0': 'SEX_FEMALE'})

SEX
2.0    6162
1.0    3838
Name: count, dtype: int64


The `value_counts()` function was first used to examine the distribution of the `'SEX'` variable, showing:
- `2.0` = Female with 6,162 clients  
- `1.0` = Male with 3,838 clients  

This gives a clear understanding of the gender breakdown in the dataset.

I then applied `pd.get_dummies()` with `drop_first=True`, which avoids multicollinearity by generating only one dummy variable. Since the original values are `1 = male` and `2 = female`, this created a column named `'SEX_2'`, corresponding to clients who are female. To match the assignment instructions and clearly reflect the category it represents, I renamed `SEX_2` to `SEX_FEMALE`.

Meaning of the new variable `SEX_FEMALE`:
- `SEX_FEMALE = 1` means the client is female
- `SEX_FEMALE = 0` means the client is male


This transformation ensures the gender feature is in a numeric format suitable for machine learning models, while retaining its interpretability. I also used `dtype=int` to make sure the dummy values are stored as `0` and `1`, rather than Boolean values.

In [8]:
# print value_count of MARRIAGE column
marriage_freq = df_cleaned['MARRIAGE'].value_counts()
marriage_freq

MARRIAGE
2.0    5518
1.0    4380
3.0      82
0.0      20
Name: count, dtype: int64

According to the dataset definition, the `MARRIAGE` variable should only contain three valid categories:

- `1` = Married
- `2` = Single
- `3` = Others

However, the `value_counts()` output shows that there are also 20 entries are coded as `0`, which is not defined in the original variable description. This suggests a discrepancy between the dataset and its documentation, suggesting a data quality issue. These undefined values may result from input errors or inconsistent coding and must be addressed during preprocessing to prevents potential bias or misleading during model training.

Most clients are either single (5,518) or married (4,380), while only 82 are classified as “others.” Since the `0` values are undefined and fewer than category `3`, a practical solution is to reassign them to category `3` (Others) to preserve all data without introducing an invalid class.


In [9]:
# reassign the abnormal value '0' to '3' = Other
df_cleaned.loc[df_cleaned['MARRIAGE'] == 0, 'MARRIAGE'] = 3

# add dummy variable for MARRIAGE column to the dataframe
df_cleaned = pd.get_dummies(df_cleaned, columns = ['MARRIAGE'], dtype = int, drop_first =True)

# rename dummy columns
df_cleaned = df_cleaned.rename(columns = {
    'MARRIAGE_2.0': 'MARRIAGE_SINGLE', 
    'MARRIAGE_3.0': 'MARRIAGE_OTHER'})

Before encoding MARRIAGE feature, we replaced the abnormal value `0` to category `3` (Others).

I applied `pd.get_dummies()` to the `MARRIAGE` column to convert it into binary variables. The `dtype=int` ensures that the resulting values are stored as integers (`0` and `1`), which is preferred by most machine learning algorithms over boolean types. "drop_first = True" is used to drop the first new variable `MARRIAGE_1.0`, which represents "Married" to avoid multicollinearity.

I renamed the columns using `.rename()` as follows:

- `MARRIAGE_2.0` to `IS_SINGLE`  
- `MARRIAGE_3.0` to `IS_OTHER` 

Each of these variables follows the binary format (`0` or `1`) and represents:

- `IS_SINGLE` = 1 if the client is single, 0 otherwise  
- `IS_OTHER` = 1 if the client belongs to the "Others" category
- When both values in `IS_SINGLE` and `IS_OTHER` are 0, it indicates that the marriage status is married (`MARRIAGE` = `1`).

In [10]:
# print value_count of MARRIAGE column
edu_freq = df_cleaned['EDUCATION'].value_counts()
edu_freq

EDUCATION
2.0    4901
1.0    3415
3.0    1543
5.0      79
4.0      43
6.0      13
0.0       6
Name: count, dtype: int64

The `EDUCATION` feature originally contained several undocumented or redundant categories (`0`, `5`, and `6`). 

According to the dataset documentation, categories `4`, `5`, and `6` all represent 'Others' or 'Unknown.' To simplify the model and improve the quality of the data, I consolidated these into a single 'Others' category (represented by the value `4`). By grouping these sparse, unlabelled values together, I reduced noise in the feature and ensured that the model treats all non-standard education levels as a single group.

In [11]:
# convert the values {0, 5, 6} to the value 4 In the column 'EDUCATION'
df_cleaned.loc[df_cleaned['EDUCATION'].isin([0, 5, 6]), 'EDUCATION'] = 4

# add dummy variable for EDUCATION column to the dataframe
df_cleaned = pd.get_dummies(df_cleaned, columns=['EDUCATION'], dtype=int, drop_first=True)

Similar logic to `MARRIGE` variable, I also applied `pd.get_dummies` for `EDUCATION` varible.

After encoding, each of these variables follows the binary format (`0` or `1`) and represents:

- `EDUCATION_2.0` = 1 if the client's education belongs to university category
- `EDUCATION_3.0` = 1 if the client's education belongs to high school category
- `EDUCATION_4.0` = 1 if the client's education belongs to "Other"category

When all values in `EDUCATION_2.0`, `EDUCATION_3.0` and `EDUCATION_4.0` are 0, it indicates that the education status is graduate school (`EDUCATION` = `1`).

## 3. Training Models

### 3.1 Prepare train and test dataset

In [12]:
# Create a numpy array `y` from the first 7,000 observations of `default payment next month` column from `df_cleaned` dataframe
y = df_cleaned["default payment next month"].loc[:6999]

# Create a numpy array `X`  from the first 7,000 observations of all the remaining variables in `df_cleaned` dataframe
X = df_cleaned.drop(["default payment next month"], axis = 1).loc[:6999]

In [13]:
# Split data to training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 31, stratify = y)

# Standardise the data to mean zero and variance one using an approapriate `sklearn` library
sc = StandardScaler()

# Fit and transform thge training data
X_train_scaled = sc.fit_transform(X_train)

# Transform the testing data
X_test_scaled = sc.transform(X_test)

### 3.2 Training Models and Interpretation


#### Logistic Regression Model

In [14]:
# Train Logistic Regression model
lr = LogisticRegression(C=100.0, random_state=10, solver='lbfgs')
lr.fit(X_train_scaled, y_train)

# Accuracy of training dataset
lr_accuracy_train = lr.score(X_train_scaled, y_train)
print(f'Accuracy Training Dataset: Logistic Regresssion = {lr_accuracy_train:.3f}')

# Accuracy of test dataset
lr_accuracy_test = lr.score(X_test_scaled, y_test)
print(f'Accuracy Test Dataset - Logistic Regresssion = {lr_accuracy_test:.3f}')

Accuracy Training Dataset: Logistic Regresssion = 0.820
Accuracy Test Dataset - Logistic Regresssion = 0.804


#### K-nearest neighbors (KNN) Model

In [15]:
# Train K-nearest neighbors (KNN) model
knn = KNeighborsClassifier(n_neighbors=5, p=2, metric='minkowski')
knn.fit(X_train_scaled, y_train)

# Accuracy of training dataset
knn_train = knn.score(X_train_scaled, y_train)
print(f'Accuracy Training Dataset - KNN = {knn_train:.3f}')

# Accuracy of test dataset
knn_test = knn.score(X_test_scaled, y_test)
print(f'Accuracy Test Dataset - KNN = {knn_test:.3f}')

Accuracy Training Dataset - KNN = 0.842
Accuracy Test Dataset - KNN = 0.784


#### Final Validation

In [ ]:
# extract the unused data (rows 7,000 to 9,999)
y_val = df_cleaned["default payment next month"].loc[7000:]
X_val = df_cleaned.drop(["default payment next month"], axis=1).loc[7000:]

# scale the validation features
X_val_scaled = sc.transform(X_val)

# final Validation for Logistic Regression
lr_val_score = lr.score(X_val_scaled, y_val)
print(f"Final Validation Accuracy (Logistic Regression): {lr_val_score:.3f}")

# final Validation for KNN
knn_val_score = knn.score(X_val_scaled, y_val)
print(f"Final Validation Accuracy (KNN): {knn_val_score:.3f}")

Final Validation Accuracy (Logistic Regression): 0.808
Final Validation Accuracy (KNN): 0.782


**Comparision between two models**

- The Logistic Regression model achieved a training accuracy of 0.820 and a test accuracy of 0.804 with a very small gap. This indicates that the model has learned general patterns that apply to new, unseen data. 

- In contrast, K-nearest Neighbor (KNN) model has a higher training accuracy of 0.842 but a lower test accuracy of 0.784. This suggests that KNN has a little overfitting in training data, which often capture local patterns but fail to predict on unseen data. This behaviour is typical of memory-based, nonlinear models such as KNN.

- Logistic Regression accuracy on final validation set of 0.808 is close to the test score (0.804) and higher than final validation accuracy of KNN (0.782), it confirms Logistic Regression model is highly stable and ready for use.

**Model Recommendation**

- Based on the performance, I would recommend using Logistic Regression to predict credit card default.Despite its slightly lower training accuracy, it achieved better performance on the test set, indicating more reliable predictions on new data, a key goal in credit risk modeling.

- Moreover, logistic regression is also highly interpretable, which is valuable in risk assessments and in financial contexts where decisions must be explainable and transparent to stakeholders. Its coefficients can provide insights into which variables most strongly influence default risk, which help support decision-making.

- KNN may still be useful in specific scenarios where the goal is to capture nonlinear patterns, but in this case, its performance suggests it is less suited for the given dataset without further tuning or feature engineering.

- Finally, this project goes beyond traditional binary classification, which simply labels clients as credible or not. Instead, it focuses on identifying exactly who is most likely to default, allowing for more targeted risk management. Among the models tested, logistic regression offered the best balance between performance, interpretability and robustness, making it highly suitable for real-world financial applications, where both accuracy and explainability are essential.

---
---